# 23.2 Extensive-form & sequential games

When actions unfold over time, solve the future first: credible plans survive backward induction. This lesson adds time to normal-form games: strategies must include off-path decisions, dynamic programming backs values up from terminal nodes, and information sets use belief-weighted payoffs. Save a copy to Drive to edit.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

SEED = 232
random.seed(SEED)
np.random.seed(SEED)

## The concept, built once: backward induction

In an extensive-form game, solve terminal decisions first. The entry-game lesson has Out $(0,2)$, Fight $(-1,-1)$, and Accommodate $(2,1)$. Backward induction removes Fight because the incumbent payoff $1$ for Accommodate is greater than $-1$ for Fight.

In [ ]:
def terminal(payoff):
    return {"terminal": True, "payoff": tuple(payoff)}


def decision(player, actions):
    return {"terminal": False, "player": player, "actions": actions}


def backward_induction(node):
    if node["terminal"]:
        return node["payoff"], [], {"terminal": node["payoff"]}

    player = node["player"]
    candidates = []
    child_values = {}
    for action, child in node["actions"].items():
        payoff, path, values = backward_induction(child)
        candidates.append((payoff[player], action, payoff, path, values))
        child_values[action] = payoff

    candidates.sort(key=lambda item: (-item[0], item[1]))
    chosen_value, chosen_action, chosen_payoff, chosen_path, chosen_child_values = candidates[0]
    path = [(player, chosen_action)] + chosen_path
    values = {"player": player, "choice": chosen_action, "children": child_values, "payoff": chosen_payoff}
    return chosen_payoff, path, values


entry_tree = decision(0, {
    "Out": terminal((0, 2)),
    "Enter": decision(1, {
        "Fight": terminal((-1, -1)),
        "Accommodate": terminal((2, 1)),
    }),
})
entry_payoff, entry_path, entry_values = backward_induction(entry_tree)
print(entry_path, entry_payoff)
assert entry_path == [(0, "Enter"), (1, "Accommodate")]
assert entry_payoff == (2, 1)

Information sets are not pointwise maxima. With beliefs $(0.6,0.4)$, Left pays $0.6\cdot3+0.4\cdot0=1.800000$ and Right pays $0.6\cdot1+0.4\cdot2=1.400000$, so Left is the belief-weighted best action.

In [ ]:
def belief_weighted_choice(beliefs, action_payoffs):
    beliefs = np.asarray(beliefs, dtype=float)
    values = {}
    for action, payoffs in action_payoffs.items():
        payoffs = np.asarray(payoffs, dtype=float)
        values[action] = float(np.dot(beliefs, payoffs))
    best_action = max(values, key=lambda action: values[action])
    return best_action, values


info_action, info_values = belief_weighted_choice(
    [0.6, 0.4],
    {"Left": [3, 0], "Right": [1, 2]},
)
print(info_action, info_values)
assert info_action == "Left"
assert round(info_values["Left"], 6) == 1.8
assert round(info_values["Right"], 6) == 1.4
assert 2 * 0.5 == 1.0

## The dataset ladder: D1 to D5 sequential profiles

The ladder grows from a tiny entry tree to deeper multi-player and imperfect-information trees with non-credible threats.

In [ ]:
def build_sequential_ladder():
    d1 = entry_tree
    d2 = decision(0, {
        "Reject": terminal((0, 0)),
        "Offer": decision(1, {
            "Accept": terminal((2, 2)),
            "Counter": decision(0, {
                "AcceptCounter": terminal((1, 3)),
                "Walk": terminal((0, 0)),
            }),
        }),
    })
    d3 = decision(0, {
        "StayOut": terminal((0, 2)),
        "Enter": decision(1, {
            "Punish": terminal((-2, -2)),
            "Share": terminal((3, 1)),
        }),
    })
    d4 = decision(0, {
        "Small": decision(1, {
            "Support": decision(2, {
                "Approve": terminal((2, 2, 1)),
                "Block": terminal((0, 1, 2)),
            }),
            "Delay": terminal((1, 3, 0)),
        }),
        "Large": decision(1, {
            "Support": decision(2, {
                "Approve": terminal((4, 1, 3)),
                "Block": terminal((-1, 0, 4)),
            }),
            "Delay": terminal((0, 2, 1)),
        }),
    })
    d5 = decision(0, {
        "EnterA": decision(1, {
            "Fight": terminal((-3, -2)),
            "Accommodate": decision(0, {
                "Expand": terminal((5, 1)),
                "Hold": terminal((3, 2)),
            }),
        }),
        "EnterB": decision(1, {
            "Fight": terminal((-2, -3)),
            "Accommodate": terminal((4, 2)),
        }),
        "Out": terminal((0, 3)),
    })
    return [
        {"name": "D1 entry", "tree": d1, "nodes": 3},
        {"name": "D2 bargaining", "tree": d2, "nodes": 5},
        {"name": "D3 off-path threat", "tree": d3, "nodes": 3},
        {"name": "D4 three-player rollout", "tree": d4, "nodes": 7},
        {"name": "D5 imperfect-threat tree", "tree": d5, "nodes": 8},
    ]


sequential_ladder = build_sequential_ladder()
for rung in sequential_ladder:
    print(rung["name"], "nodes", rung["nodes"])

## Run the SAME method across D1-D5

Use backward induction on every tree and collect root value plus removed non-credible continuations.

In [ ]:
def count_removed_threats(node):
    if node["terminal"]:
        return 0
    player = node["player"]
    action_payoffs = []
    total = 0
    for action, child in node["actions"].items():
        payoff, path, values = backward_induction(child)
        action_payoffs.append(payoff[player])
        total += count_removed_threats(child)
    best = max(action_payoffs)
    total += sum(1 for value in action_payoffs if value < best)
    return total


seq_results = []
for rung in sequential_ladder:
    payoff, path, values = backward_induction(rung["tree"])
    removed = count_removed_threats(rung["tree"])
    seq_results.append({
        "name": rung["name"],
        "nodes": rung["nodes"],
        "payoff0": payoff[0],
        "removed": removed,
        "path": path,
    })

for row in seq_results:
    print(row["name"], "payoff0", row["payoff0"], "removed", row["removed"], "path", row["path"])

## Results visualization

The closing figure has game-tree/path panels plus value and credibility curves over tree size.

In [ ]:
fig, axes = plt.subplots(2, len(seq_results), figsize=(18, 7))

for axis, row in zip(axes[0], seq_results):
    text = row["name"] + "\n"
    text += "path " + " -> ".join(action for player, action in row["path"]) + "\n"
    text += "root payoff " + str(row["payoff0"])
    axis.text(0.05, 0.5, text, fontsize=10, va="center")
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.axis("off")

nodes = [row["nodes"] for row in seq_results]
values = [row["payoff0"] for row in seq_results]
removed = [row["removed"] for row in seq_results]
axes[1, 0].plot(nodes, values, marker="o", label="root player value")
axes[1, 0].set_xlabel("tree nodes")
axes[1, 0].set_ylabel("value")
axes[1, 0].legend()
axes[1, 1].plot(nodes, removed, marker="s", color="tab:red", label="inferior continuations removed")
axes[1, 1].set_xlabel("tree nodes")
axes[1, 1].set_ylabel("count")
axes[1, 1].legend()
for axis in axes[1, 2:]:
    axis.axis("off")
plt.tight_layout()
plt.show()

## Pitfall on D5: keeping non-credible threats

The wrong profile lets the incumbent threaten Fight after entry. Backward induction checks the subgame and removes that threat when Accommodate gives the incumbent more.

In [ ]:
d5_tree = sequential_ladder[-1]["tree"]
wrong_path = [(0, "Out")]
wrong_value = 0
fixed_payoff, fixed_path, fixed_values = backward_induction(d5_tree)
print("wrong path held by threat", wrong_path, "entrant value", wrong_value)
print("fixed path", fixed_path, "entrant value", fixed_payoff[0])
assert fixed_payoff[0] > wrong_value

## Evaluate it + Practice

- Metric: solved root value and number of inferior continuations removed; compare with a no-skill policy that chooses the first listed action.
- Sanity check: every chosen action must maximize the current player's backed-up continuation payoff.
- Ablation: if you skip off-path nodes, D3 and D5 keep harmful threats.
- Failure signal: optimizing separately inside an information set contradicts the single-action requirement.

Practice 1: Add a second bargaining counteroffer and rerun backward induction.

Practice 2: Change the belief in the information set from 0.6 to 0.3 and find the chosen action.